In [31]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq                 # correct Groq LLM import
from dotenv import load_dotenv 
from langgraph.checkpoint.memory import MemorySaver

In [32]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [33]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


In [34]:

def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [35]:
checkpointer=MemorySaver()


graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [36]:

memory = MemorySaver()

# ! compile with checkpointer
chatbot = graph.compile(checkpointer=memory)

In [37]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
thread_id = '1'

while True:
    user_message = input('Type here: ')

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    # ! config for persistence (thread memory)
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = chatbot.invoke(
        {
            'messages': [HumanMessage(content=user_message)]
        },
        config=config
    )

    print('AI:', response['messages'][-1].content)

In [ ]:
chatbot.get_state(config=config)